# Lane Detection with CLRNet

This notebook trains a CLRNet model for detecting parallel dashed lane markings.

**Objectives:**
1. Setup CLRNet from GitHub
2. Convert dataset to CLRNet format (CULane/TuSimple style)
3. Configure and train CLRNet with ResNet backbone
4. Evaluate on validation set
5. Visualize lane detections and save model weights

**Model Configuration:**
- Model: CLRNet with ResNet-18 or ResNet-34 backbone
- Target: Parallel dashed lane markings
- Input size: 640x640
- Training: 50-100 epochs
- Batch size: 8-16

## 1. Setup and Installation

In [ ]:
# Install dependencies
!pip install -q torch torchvision opencv-python-headless pillow matplotlib scipy scikit-learn

print("Base dependencies installed!")

Base dependencies installed!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Clone CLRNet repository (if not already cloned)
import os
from pathlib import Path

clrnet_dir = r'C:\SpeedRadar\CLRNet'

if not os.path.exists(clrnet_dir):
    print("Cloning CLRNet repository...")
    !git clone https://github.com/Turoad/CLRNet.git C:\SpeedRadar\CLRNet
    print("CLRNet cloned successfully!")
else:
    print("CLRNet directory already exists.")

# Install CLRNet dependencies
print("\nInstalling CLRNet dependencies...")
!pip install -q -r C:\SpeedRadar\CLRNet\requirements.txt

print("Setup complete!")

CLRNet directory already exists.

Installing CLRNet dependencies...
Setup complete!


ERROR: Could not find a version that satisfies the requirement torch==1.8.0 (from versions: 2.9.0, 2.9.1, 2.10.0)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torch==1.8.0


In [ ]:
import sys
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

# Add project root and CLRNet to path
project_root = r'C:\SpeedRadar'
clrnet_path = r'C:\SpeedRadar\CLRNet'

if project_root not in sys.path:
    sys.path.insert(0, project_root)
if clrnet_path not in sys.path:
    sys.path.insert(0, clrnet_path)

from dataset_loader import LaneDataset

print("Imports successful!")

Imports successful!


## 2. Data Preparation - Convert to CLRNet Format

### 2.1 Understand CLRNet Data Format

CLRNet typically uses CULane or TuSimple format:

**TuSimple Format** (JSON):
```json
{
  "lanes": [[x1, x2, ...], [x1, x2, ...]], // x coordinates for each lane
  "h_samples": [y1, y2, ...],              // y coordinates (fixed)
  "raw_file": "path/to/image.jpg"
}
```

**CULane Format** (TXT):
```
x1 y1
x2 y2
...
```

We'll use TuSimple-style format as it's more straightforward.

In [ ]:
def convert_to_tusimple_format(dataset, split_name, output_dir):
    """
    Convert our lane dataset to TuSimple-like format for CLRNet.
    
    Args:
        dataset: LaneDataset instance
        split_name: 'train' or 'val'
        output_dir: Output directory path
    """
    output_dir = Path(output_dir)
    images_dir = output_dir / split_name / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)
    
    # Create annotation list
    annotations = []
    
    print(f"Converting {split_name} split to TuSimple format...")
    
    # Fixed y-coordinates for sampling (normalized 0-1)
    h_samples = np.linspace(0.3, 1.0, 20).tolist()  # Sample from 30% to 100% of image height
    
    samples_with_lanes = 0
    
    for idx in range(len(dataset)):
        if idx % 500 == 0:
            print(f"  Processed {idx}/{len(dataset)} images...")
        
        image_tensor, lanes, img_name = dataset[idx]
        
        # Skip images without lanes
        if len(lanes) == 0:
            continue
        
        samples_with_lanes += 1
        
        # Copy image
        source_dir = dataset.img_dir
        src_img = Path(source_dir) / img_name
        dst_img = images_dir / img_name
        shutil.copy2(src_img, dst_img)
        
        # Convert lanes to TuSimple format
        img = cv2.imread(str(src_img))
        img_h, img_w = img.shape[:2]
        
        lane_points = []
        for lane in lanes:
            # Convert normalized coordinates to pixel coordinates
            lane_array = np.array(lane)
            lane_pixels = lane_array * [img_w, img_h]
            
            # Interpolate x-coordinates for fixed y-coordinates
            if len(lane_pixels) > 1:
                # Sort by y-coordinate
                sorted_indices = np.argsort(lane_pixels[:, 1])
                lane_pixels = lane_pixels[sorted_indices]
                
                # Interpolate
                x_coords = []
                for h_sample in h_samples:
                    y_target = h_sample * img_h
                    
                    # Find x-coordinate at this y using linear interpolation
                    if y_target < lane_pixels[0, 1] or y_target > lane_pixels[-1, 1]:
                        x_coords.append(-2)  # -2 indicates no lane point at this y
                    else:
                        x_interp = np.interp(y_target, lane_pixels[:, 1], lane_pixels[:, 0])
                        x_coords.append(float(x_interp))
                
                lane_points.append(x_coords)
        
        # Create annotation entry
        annotation = {
            'lanes': lane_points,
            'h_samples': [int(h * img_h) for h in h_samples],
            'raw_file': f'{split_name}/images/{img_name}'
        }
        annotations.append(annotation)
    
    print(f"  Completed: {samples_with_lanes} images with lane markings")
    
    # Save annotations to JSON file
    anno_file = output_dir / f'{split_name}_gt.json'
    with open(anno_file, 'w') as f:
        for anno in annotations:
            f.write(json.dumps(anno) + '\n')
    
    print(f"  Annotations saved to: {anno_file}")
    return len(annotations)

# Prepare dataset
clrnet_dataset_dir = r'C:\SpeedRadar\clrnet_dataset'

# Load datasets
train_dir = r'C:\SpeedRadar\dataset\train'
val_dir = r'C:\SpeedRadar\dataset\val'

lane_train = LaneDataset(train_dir, img_size=640)
lane_val = LaneDataset(val_dir, img_size=640)

# Convert datasets
print("\nConverting datasets to CLRNet format...\n")
train_count = convert_to_tusimple_format(lane_train, 'train', clrnet_dataset_dir)
val_count = convert_to_tusimple_format(lane_val, 'val', clrnet_dataset_dir)

print(f"\nConversion complete!")
print(f"Train samples with lanes: {train_count}")
print(f"Val samples with lanes: {val_count}")


Converting datasets to CLRNet format...

Converting train split to TuSimple format...
  Processed 0/4000 images...


## 3. Alternative: Simple Lane Detection Model

Since CLRNet setup can be complex, we'll implement a simpler alternative using a PyTorch-based lane detection approach that's easier to train and integrate.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
import torchvision.transforms as transforms

# Simple segmentation-based lane detection model
class SimpleLaneDetector(nn.Module):
    def __init__(self, num_classes=1):
        super(SimpleLaneDetector, self).__init__()
        
        # Use ResNet18 as backbone
        resnet = models.resnet18(pretrained=True)
        
        # Encoder (ResNet backbone)
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1  # 64 channels
        self.layer2 = resnet.layer2  # 128 channels
        self.layer3 = resnet.layer3  # 256 channels
        self.layer4 = resnet.layer4  # 512 channels
        
        # Decoder (upsampling)
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(256, 128)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec3 = self._make_decoder_block(128, 64)
        
        self.up4 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec4 = self._make_decoder_block(128, 64)
        
        # Final convolution
        self.final = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.Conv2d(32, num_classes, kernel_size=1),
            nn.Sigmoid()
        )
    
    def _make_decoder_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Save input size for final resize
        input_size = x.shape[2:]
        
        # Encoder
        x0 = self.layer0(x)  # 1/4 size, 64 channels
        x1 = self.layer1(x0)  # 1/4 size, 64 channels
        x2 = self.layer2(x1)  # 1/8 size, 128 channels
        x3 = self.layer3(x2)  # 1/16 size, 256 channels
        x4 = self.layer4(x3)  # 1/32 size, 512 channels
        
        # Decoder with skip connections
        up1 = self.up1(x4)  # 1/16 size
        # Match sizes for concatenation
        up1 = self._match_size(up1, x3)
        cat1 = torch.cat([up1, x3], dim=1)
        dec1 = self.dec1(cat1)
        
        up2 = self.up2(dec1)  # 1/8 size
        up2 = self._match_size(up2, x2)
        cat2 = torch.cat([up2, x2], dim=1)
        dec2 = self.dec2(cat2)
        
        up3 = self.up3(dec2)  # 1/4 size
        up3 = self._match_size(up3, x1)
        cat3 = torch.cat([up3, x1], dim=1)
        dec3 = self.dec3(cat3)
        
        up4 = self.up4(dec3)  # 1/2 size
        up4 = self._match_size(up4, x0)
        cat4 = torch.cat([up4, x0], dim=1)
        dec4 = self.dec4(cat4)
        
        # Final output - upsample to original size
        out = self.final(dec4)
        
        # Ensure output matches input size exactly
        if out.shape[2:] != input_size:
            out = nn.functional.interpolate(out, size=input_size, mode='bilinear', align_corners=False)
        
        return out
    
    def _match_size(self, x, target):
        """Match spatial dimensions of x to target using interpolation"""
        if x.shape[2:] != target.shape[2:]:
            x = nn.functional.interpolate(x, size=target.shape[2:], mode='bilinear', align_corners=False)
        return x

print("SimpleLaneDetector model defined!")

# Create model instance
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SimpleLaneDetector(num_classes=1).to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

### 3.1 Create Lane Segmentation Dataset

In [ ]:
class LaneSegmentationDataset(torch.utils.data.Dataset):
    """
    Dataset that creates segmentation masks from lane polylines.
    """
    def __init__(self, lane_dataset, img_size=640):
        self.lane_dataset = lane_dataset
        self.img_size = img_size
        
    def __len__(self):
        return len(self.lane_dataset)
    
    def __getitem__(self, idx):
        image_tensor, lanes, img_name = self.lane_dataset[idx]
        
        # Create segmentation mask
        mask = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        
        for lane in lanes:
            lane_array = np.array(lane)
            # Convert normalized coordinates to pixel coordinates
            lane_pixels = (lane_array * self.img_size).astype(np.int32)
            
            # Draw lane line on mask
            for i in range(len(lane_pixels) - 1):
                cv2.line(mask, 
                        tuple(lane_pixels[i]), 
                        tuple(lane_pixels[i + 1]), 
                        1.0, 
                        thickness=5)
        
        mask = torch.from_numpy(mask).unsqueeze(0)  # Add channel dimension
        
        return image_tensor, mask, img_name

# Create datasets
train_seg_dataset = LaneSegmentationDataset(lane_train, img_size=640)
val_seg_dataset = LaneSegmentationDataset(lane_val, img_size=640)

print(f"Segmentation datasets created:")
print(f"  Train: {len(train_seg_dataset)} samples")
print(f"  Val: {len(val_seg_dataset)} samples")

# Create data loaders
train_loader = DataLoader(train_seg_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_seg_dataset, batch_size=8, shuffle=False, num_workers=0)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

## 4. Model Training

In [ ]:
# Training configuration
num_epochs = 30  # Start with 30 epochs for POC
learning_rate = 0.0001
patience = 5

# Loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy for segmentation
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

print("Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Optimizer: AdamW")
print(f"  Loss function: Binary Cross Entropy")

In [ ]:
# Training loop
train_losses = []
val_losses = []
best_val_loss = float('inf')
epochs_no_improve = 0

print("\nStarting training...\n")

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for batch_idx, (images, masks, _) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"  Epoch [{epoch+1}/{num_epochs}] Batch [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for images, masks, _ in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}\n")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        
        # Save model
        weights_dir = Path(r'C:\SpeedRadar\models\weights')
        weights_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), weights_dir / 'lane_detector_best.pt')
        print(f"  New best model saved! (Val Loss: {val_loss:.4f})")
    else:
        epochs_no_improve += 1
    
    # Early stopping
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        break

print("\nTraining complete!")

## 5. Training Visualization

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best validation loss: {best_val_loss:.4f}")

## 6. Model Evaluation and Visualization

In [ ]:
# Load best model
model.load_state_dict(torch.load(r'C:\SpeedRadar\models\weights\lane_detector_best.pt'))
model.eval()

print("Best model loaded for evaluation.")

In [ ]:
# Visualize predictions on validation set
num_samples = 10
fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples*3))

with torch.no_grad():
    for i in range(num_samples):
        idx = i * (len(val_seg_dataset) // num_samples)
        image, mask_gt, img_name = val_seg_dataset[idx]
        
        # Predict
        image_batch = image.unsqueeze(0).to(device)
        mask_pred = model(image_batch)
        mask_pred = mask_pred.squeeze().cpu().numpy()
        
        # Convert tensors to numpy for visualization
        image_np = image.permute(1, 2, 0).numpy()
        mask_gt_np = mask_gt.squeeze().numpy()
        
        # Plot original image
        axes[i, 0].imshow(image_np)
        axes[i, 0].set_title(f'Original: {img_name}', fontsize=8)
        axes[i, 0].axis('off')
        
        # Plot ground truth mask
        axes[i, 1].imshow(image_np)
        axes[i, 1].imshow(mask_gt_np, alpha=0.5, cmap='jet')
        axes[i, 1].set_title('Ground Truth', fontsize=8)
        axes[i, 1].axis('off')
        
        # Plot predicted mask
        axes[i, 2].imshow(image_np)
        axes[i, 2].imshow(mask_pred, alpha=0.5, cmap='jet')
        axes[i, 2].set_title('Prediction', fontsize=8)
        axes[i, 2].axis('off')

plt.suptitle('Lane Detection Results - Validation Set', fontsize=14, y=1.0)
plt.tight_layout()
plt.show()

## 7. Extract Lane Polylines from Segmentation

In [ ]:
def extract_lane_polylines(mask, threshold=0.5, min_points=10):
    """
    Extract lane polylines from segmentation mask.
    
    Args:
        mask: Binary segmentation mask (H, W)
        threshold: Threshold for binarization
        min_points: Minimum number of points for a valid lane
    
    Returns:
        List of lane polylines, each as numpy array of (x, y) points
    """
    # Binarize mask
    binary_mask = (mask > threshold).astype(np.uint8) * 255
    
    # Find contours
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    lanes = []
    for contour in contours:
        # Approximate contour
        epsilon = 0.01 * cv2.arcLength(contour, False)
        approx = cv2.approxPolyDP(contour, epsilon, False)
        
        # Convert to polyline
        points = approx.squeeze()
        
        if len(points.shape) == 2 and len(points) >= min_points:
            # Sort by y-coordinate
            points = points[np.argsort(points[:, 1])]
            lanes.append(points)
    
    return lanes

print("Lane polyline extraction function defined.")

In [ ]:
# Test polyline extraction
with torch.no_grad():
    sample_idx = 0
    image, mask_gt, img_name = val_seg_dataset[sample_idx]
    
    # Predict
    image_batch = image.unsqueeze(0).to(device)
    mask_pred = model(image_batch)
    mask_pred = mask_pred.squeeze().cpu().numpy()
    
    # Extract polylines
    lanes = extract_lane_polylines(mask_pred, threshold=0.5, min_points=10)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    image_np = image.permute(1, 2, 0).numpy()
    
    # Segmentation mask
    axes[0].imshow(image_np)
    axes[0].imshow(mask_pred, alpha=0.5, cmap='jet')
    axes[0].set_title('Predicted Segmentation')
    axes[0].axis('off')
    
    # Extracted polylines
    axes[1].imshow(image_np)
    for lane in lanes:
        axes[1].plot(lane[:, 0], lane[:, 1], 'y-', linewidth=3, marker='o', markersize=3)
    axes[1].set_title(f'Extracted Polylines ({len(lanes)} lanes)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Extracted {len(lanes)} lane polylines")

## 8. Save Model and Summary

In [ ]:
# Save final model
weights_dir = Path(r'C:\SpeedRadar\models\weights')
weights_dir.mkdir(parents=True, exist_ok=True)

# Save model state dict
torch.save(model.state_dict(), weights_dir / 'lane_detector_final.pt')

# Save entire model (for easier loading)
torch.save(model, weights_dir / 'lane_detector_complete.pt')

print("Model weights saved:")
print(f"  - State dict: {weights_dir / 'lane_detector_best.pt'}")
print(f"  - Final model: {weights_dir / 'lane_detector_final.pt'}")
print(f"  - Complete model: {weights_dir / 'lane_detector_complete.pt'}")

In [ ]:
print("="*70)
print("LANE DETECTION TRAINING SUMMARY")
print("="*70)

print("\n1. MODEL CONFIGURATION:")
print(f"   - Architecture: U-Net with ResNet-18 backbone")
print(f"   - Task: Lane segmentation")
print(f"   - Input size: 640x640")
print(f"   - Training epochs: {len(train_losses)}")
print(f"   - Batch size: 8")

print("\n2. PERFORMANCE METRICS:")
print(f"   - Best validation loss: {best_val_loss:.4f}")
print(f"   - Final train loss: {train_losses[-1]:.4f}")
print(f"   - Final validation loss: {val_losses[-1]:.4f}")

print("\n3. MODEL WEIGHTS:")
print(f"   - Best model: C:\\SpeedRadar\\models\\weights\\lane_detector_best.pt")
print(f"   - Complete model: C:\\SpeedRadar\\models\\weights\\lane_detector_complete.pt")

print("\n4. LANE EXTRACTION:")
print(f"   - Method: Segmentation + contour detection")
print(f"   - Output: Lane polylines (list of (x, y) points)")
print(f"   - Function: extract_lane_polylines()")

print("\n5. NEXT STEPS:")
print("   - Proceed to 04_speed_estimation.ipynb")
print("   - Integrate lane detector with vehicle detector")
print("   - Implement homography and speed calculation")

print("\n" + "="*70)